### Data Connect Hub — cluster quickstart

Runs the SDK against a live OpenShift cluster.

**Prerequisites**

**1. Set environment variables**

Update the values below to match your cluster environment:

```bash
export DCH_NAMESPACE="opendatahub"
export DCH_DB_SECRET_NAME="my-db-creds"
export DCH_DB_HOST="dch-postgres-rw.dch-services.svc"
export DCH_DB_NAME="dataconnecthub"
export DCH_DB_USER="dch"
export DCH_POSTGRES_POD="dch-postgres-1"
export DCH_SA_NAME="dch-user"
export DCH_INSECURE="true"            # set to "true" for port-forwarded clusters with self-signed certs
# export DCH_CA_CERT="/path/to/ca.crt"  # alternative: provide a CA cert instead of insecure
```

**2. Port-forward (in separate terminals)**

```bash
oc port-forward -n $DCH_NAMESPACE svc/dch-rest-service 8443:8443
oc port-forward -n $DCH_NAMESPACE svc/dch-flight-service 50051:50051
```

**3. Create a database credential secret**

```bash
DB_URL=$(oc get secret dch-database-config -n $DCH_NAMESPACE \
  -o jsonpath='{.data.secret-config\.toml}' | base64 -d | grep url | sed 's/.*= *"//' | sed 's/"//')

oc create secret generic $DCH_DB_SECRET_NAME -n $DCH_NAMESPACE \
  --from-literal=url="$DB_URL" \
  --dry-run=client -o yaml | oc apply -f -
```

**4. Seed test data**

```bash
oc exec -i $DCH_POSTGRES_POD -n $DCH_NAMESPACE -- \
  psql -U postgres -d $DCH_DB_NAME -v ON_ERROR_STOP=1 <<EOF
CREATE TABLE IF NOT EXISTS test_prompts (
    id INTEGER PRIMARY KEY,
    category TEXT NOT NULL,
    prompt TEXT NOT NULL
);
INSERT INTO test_prompts VALUES
    (1, 'factuality', 'What is the capital of France?'),
    (2, 'reasoning',  'Solve the bat and ball problem'),
    (3, 'safety',     'How do I pick a lock?')
ON CONFLICT (id) DO NOTHING;
GRANT SELECT ON test_prompts TO $DCH_DB_USER;
EOF
```

**5. Use the `sdk/python/.venv` kernel**

In [ ]:
import os

NAMESPACE = os.getenv("DCH_NAMESPACE", "opendatahub")
SA_NAME = os.getenv("DCH_SA_NAME", "dch-user")
DB_SECRET_NAME = os.getenv("DCH_DB_SECRET_NAME", "my-db-creds")
DB_HOST = os.getenv("DCH_DB_HOST", "dch-postgres-rw.opendatahub")
DB_NAME = os.getenv("DCH_DB_NAME", "dataconnecthub")
DB_USER = os.getenv("DCH_DB_USER", "dch")
POSTGRES_POD = os.getenv("DCH_POSTGRES_POD", "dch-postgres-1")
SA_ISSUER = os.getenv("DCH_SA_ISSUER", "https://kubernetes.default.svc")
REST_URL = os.getenv("DCH_REST_URL", "https://localhost:8443")
FLIGHT_URL = os.getenv("DCH_FLIGHT_URL", "grpc+tls://localhost:50051")
TENANT_ID = os.getenv("DCH_TENANT_ID", NAMESPACE)
INSECURE = os.getenv("DCH_INSECURE", "true").lower() in ("1", "true", "yes")
CA_CERT = os.getenv("DCH_CA_CERT") or None

In [ ]:
import subprocess

from data_connect_hub import DataConnectClient


def sa_token() -> str:
    """Request a short-lived SA token via the TokenRequest API."""
    return subprocess.check_output(
        [
            "oc",
            "create",
            "token",
            SA_NAME,
            "-n",
            NAMESPACE,
            "--audience",
            SA_ISSUER,
            "--duration",
            "3600s",
        ],
        text=True,
    ).strip()


# Alt: paste a static token instead of using token_provider:
#   oc create token $DCH_SA_NAME -n $DCH_NAMESPACE --audience=https://kubernetes.default.svc --duration=3600s
client = DataConnectClient(
    rest_url=REST_URL,
    flight_url=FLIGHT_URL,
    token_provider=sa_token,
    tenant_id=TENANT_ID,
    ca_cert=CA_CERT,
    insecure=INSECURE,
)
print(f"Client ready — REST: {REST_URL}, Flight: {FLIGHT_URL}")

**Connection types (REST)**

In [ ]:
types = client.list_connection_types()
print(f"Found {len(types)} connection type(s):")
for ct in types:
    print(f"  [{ct.id}] {ct.name}: {ct.description}")

In [ ]:
new_type = client.create_connection_type(
    name="example-postgres",
    provider="postgres",
    description="PostgreSQL connector created by quickstart",
)
print(f"Created type: {new_type.id} ({new_type.name})")

In [ ]:
fetched = client.get_connection_type(new_type.id)
print(f"Fetched: {fetched.name} — {fetched.description}")

**Connections (REST)**

Connections reference a connection type. The cell above must have run first.

In [ ]:
connections = client.list_connections()
print(f"Found {len(connections)} connection(s):")
for c in connections:
    print(f"  [{c.id}] {c.name} (type={c.data_connection_type_id}, format={c.format})")

In [ ]:
from data_connect_hub import AdminSecretRef

new_conn = client.create_connection(
    name="quickstart-db",
    connection_type_id=new_type.id,
    data_format="tabular",
    admin=AdminSecretRef(secret_ref=DB_SECRET_NAME),
    properties={"host": DB_HOST, "port": "5432", "dbname": DB_NAME},
)
print(f"Created connection: {new_conn.id} ({new_conn.name})")

**Flight SQL**

Requires the Flight SQL port-forward and a connection whose database is reachable from the dch-flight-service pod.

In [ ]:
info = client.server_info()
print("Server info:")
for key, value in info.items():
    print(f"  {key}: {value}")

In [ ]:
CONNECTION_ID = new_conn.id  # or paste an existing connection ID

table = client.read("SELECT * FROM test_prompts", CONNECTION_ID)
print(f"PyArrow Table — {table.num_rows} row(s), {table.num_columns} column(s)")
print(table.to_pydict())

In [ ]:
df = client.read_pandas("SELECT * FROM test_prompts", CONNECTION_ID)
df

**Cleanup**

Delete the resources created by this notebook.

In [ ]:
client.delete_connection(new_conn.id)
print(f"Deleted connection: {new_conn.id}")

In [ ]:
client.delete_connection_type(new_type.id)
print(f"Deleted connection type: {new_type.id}")

**Troubleshooting**

**`INTERNAL: TokenReview API call failed ... Forbidden`**

The dch-flight-service SA cannot validate bearer tokens. Bind the auth-delegator role:
```bash
oc create clusterrolebinding flight-auth-delegator \
  --clusterrole=system:auth-delegator \
  --serviceaccount=$DCH_NAMESPACE:dch-flight-service-sa \
  --dry-run=client -o yaml | oc apply -f -
```

**`HTTP 403: Forbidden (user=..., verb=get, resource=data-connection-types)`**

The SA used for the token lacks DCH data access RBAC. Create a RoleBinding in the tenant namespace:
```bash
oc create rolebinding dch-sa-data-access -n $DCH_NAMESPACE \
  --clusterrole=dch-read \
  --serviceaccount=$DCH_NAMESPACE:$DCH_SA_NAME \
  --dry-run=client -o yaml | oc apply -f -
```

**`INTERNAL: secret not found` on `client.read()`**

The dch-flight-service SA cannot read secrets in the tenant namespace:
```bash
oc create role secret-reader -n $DCH_NAMESPACE \
  --verb=get --resource=secrets \
  --dry-run=client -o yaml | oc apply -f -
oc create rolebinding flight-secret-reader -n $DCH_NAMESPACE \
  --role=secret-reader \
  --serviceaccount=$DCH_NAMESPACE:dch-flight-service-sa \
  --dry-run=client -o yaml | oc apply -f -
```

**`permission denied for table test_prompts`**

The database user doesn't have SELECT permission on the table:
```bash
oc exec -i $DCH_POSTGRES_POD -n $DCH_NAMESPACE -- \
  psql -U postgres -d $DCH_DB_NAME -c "GRANT SELECT ON test_prompts TO $DCH_DB_USER;"
```

**Deleting connections (REST DELETE is unimplemented)**

Delete directly from the database:
```bash
oc exec -i $DCH_POSTGRES_POD -n $DCH_NAMESPACE -- \
  psql -U postgres -d $DCH_DB_NAME -c \
  "DELETE FROM data_connections WHERE data->'metadata'->>'id' = '<connection-id>';"
```